# 03. Bagging Ensemble (Random Forest Regressor)
Triển khai Random Forest Regressor (`n_estimators=100`, `max_depth=16`, `n_jobs=-1`).


In [1]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
if not os.path.exists(PROCESSED_DIR):
    PROCESSED_DIR = os.path.join(BASE_DIR, "data_processed")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")

def load_data(scenario):
    p_train = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_train.parquet")
    p_val = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_val.parquet")
    p_test = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_test.parquet")
    if os.path.exists(p_train):
        return pd.read_parquet(p_train), pd.read_parquet(p_val), pd.read_parquet(p_test)
    return pd.read_csv(p_train.replace(".parquet", ".csv")), pd.read_csv(p_val.replace(".parquet", ".csv")), pd.read_csv(p_test.replace(".parquet", ".csv"))

def calculate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    
    act_150 = (y_true > 150)
    pred_150 = (y_pred > 150)
    tp_150 = np.sum(act_150 & pred_150)
    recall_150 = tp_150 / (np.sum(act_150) + 1e-8)
    prec_150 = tp_150 / (np.sum(pred_150) + 1e-8)
    f1_150 = 2 * prec_150 * recall_150 / (prec_150 + recall_150 + 1e-8)
    return {
        "R2": float(r2), "MAE": float(mae), "RMSE": float(rmse),
        "sMAPE": float(smape), "Recall_150": float(recall_150),
        "Precision_150": float(prec_150), "F1_150": float(f1_150)
    }


### Huấn Luyện Random Forest


In [2]:
rf_results = []

for sc in ["1h", "24h"]:
    print(f"--- Random Forest: {sc.upper()} ---")
    train_df, val_df, test_df = load_data(sc)
    feat_cols = [c for c in train_df.columns if c not in ['time', 'year', 'target']]
    
    station_map = {sid: idx for idx, sid in enumerate(train_df['station_id'].unique())}
    X_train = train_df[feat_cols].copy()
    X_val = val_df[feat_cols].copy()
    X_test = test_df[feat_cols].copy()
    
    X_train['station_id'] = X_train['station_id'].map(station_map)
    X_val['station_id'] = X_val['station_id'].map(station_map).fillna(0)
    X_test['station_id'] = X_test['station_id'].map(station_map).fillna(0)
    
    y_train = train_df['target'].values
    y_val = val_df['target'].values
    y_test = test_df['target'].values
    
    t0 = time.time()
    rf = RandomForestRegressor(
        n_estimators=100, max_depth=16, max_features=0.8,
        n_jobs=-1, random_state=42
    )
    rf.fit(X_train, y_train)
    time_rf = time.time() - t0
    
    val_pred = rf.predict(X_val)
    test_pred = rf.predict(X_test)
    
    m_val = calculate_metrics(y_val, val_pred)
    m_test = calculate_metrics(y_test, test_pred)
    
    joblib.dump(rf, f"{ARTIFACTS_DIR}/models/model_rf_{sc}.joblib")
    
    pred_df = test_df[['time', 'station_id', 'target']].copy().rename(columns={'target': 'actual'})
    pred_df['pred_random_forest'] = test_pred
    pred_df.to_csv(f"{ARTIFACTS_DIR}/predictions/preds_{sc}_random_forest.csv", index=False)
    
    rf_results.append({
        "scenario": sc, "family": "Bagging Trees", "model": "Random Forest",
        "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
        "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
        "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
        "test_F1_150": m_test["F1_150"], "time_sec": time_rf
    })
    print(f"Kịch bản {sc} ({time_rf:.1f}s): Test R2 = {m_test['R2']:.4f}, Test MAE = {m_test['MAE']:.2f}")

df_rf_res = pd.DataFrame(rf_results)
df_rf_res.to_json(f"{ARTIFACTS_DIR}/metrics/metrics_random_forest.json", orient="records", indent=2)
df_rf_res[['scenario', 'model', 'val_R2_2024', 'test_R2_2025', 'test_MAE_2025', 'test_RMSE_2025', 'time_sec']]


--- Random Forest: 1H ---


Kịch bản 1h (9.5s): Test R2 = 0.9539, Test MAE = 3.33
--- Random Forest: 24H ---


Kịch bản 24h (8.9s): Test R2 = 0.6037, Test MAE = 11.53


,scenario,model,val_R2_2024,test_R2_2025,test_MAE_2025,test_RMSE_2025,time_sec
0,1h,Random Forest,0.900061,0.953889,3.330299,5.686149,9.494799
1,24h,Random Forest,0.541863,0.603719,11.532435,16.627222,8.930093
